# Welcome to the OpenAI Agent SDK ERISA Tutorial!

The OpenAI Agents SDK enables you to build agentic AI apps in a lightweight, easy-to-use package with very few abstractions. It's a production-ready upgrade of our previous experimentation for agents, Swarm. The Agents SDK has a very small set of primitives:

Agents, which are LLMs equipped with instructions and tools
Handoffs, which allow agents to delegate to other agents for specific tasks
Guardrails, which enable validation of agent inputs and outputs
Sessions, which automatically maintains conversation history across agent runs
In combination with Python, these primitives are powerful enough to express complex relationships between tools and agents, and allow you to build real-world applications without a steep learning curve. In addition, the SDK comes with built-in tracing that lets you visualize and debug your agentic flows, as well as evaluate them and even fine-tune models for your application.

Why use the Agents SDK
The SDK has two driving design principles:

Enough features to be worth using, but few enough primitives to make it quick to learn.
Works great out of the box, but you can customize exactly what happens.

### Here are the main features of the SDK:

* **Agent loop**: Built-in agent loop that handles calling tools, sending results to the LLM, and looping until the LLM is done.
* **Python-first**: Use built-in language features to orchestrate and chain agents, rather than needing to learn new abstractions.
* **Handoffs**: A powerful feature to coordinate and delegate between multiple agents.
* **Guardrails**: Run input validations and checks in parallel to your agents, breaking early if the checks fail.
* **Sessions**: Automatic conversation history management across agent runs, eliminating manual state handling.
* **Function tools**: Turn any Python function into a tool, with automatic schema generation and Pydantic-powered validation.
* **Tracing**: Built-in tracing that lets you visualize, debug and monitor your workflows, as well as use the OpenAI suite of evaluation, fine-tuning and distillation tools.

## Important Note on Vibe Coding
Vibe coding can be a powerful accelerator. It is excellent for scaffolding, exploration, and rapidly testing ideas. Used intentionally, it can save significant time.

However, **delegating architectural decisions to vibe coding guarantees suboptimal results**. In practice, this means:
* Defaulting to older or overly generic patterns (e.g., Chat Completions when Responses is the better fit)
* Missing newer SDK abstractions
* Accumulating unnecessary glue code that could have been avoided with a deeper understanding of the framework

Agent-based systems, in particular, reward explicit design choices. Model selection, API surface (Chat Completions vs Responses), memory strategy, and tool boundaries are architectural decisions—not autocomplete tasks.

If you choose to vibe code:

* Do it *after* you understand the package and its primitives
* Provide clear, custom instructions (what to use, what to avoid, why. Cursor/Claude code even have ```instructions.md```, and you'd be a fool not to use it!)
* Constrain it with **documentation links**, **version context**, and **explicit goals**
* Treat generated code as a draft, not an authority.

Used with intent and context, vibe coding is a multiplier. Used as a substitute for understanding, it quietly locks you into mediocrity and tech debt.

## Installation

In [ ]:
pip install openai openai-agents pydantic

Note: you may need to restart the kernel to use updated packages.


## Getting Started With Agent SDK + Ollama

This section covers the first decision you need to make when wiring Ollama into the OpenAI Agents SDK: which OpenAI-style API surface to target. Ollama supports both ```/v1/chat/completions``` and ```/v1/responses```. Both can work with agent orchestration, but they differ in how they represent outputs (text, tool calls, etc.) and what the server can manage for you.

### 1. **Choose an API Surface**: Chat Completions vs. Responses
#### **Chat Completions** (```/v1/chat/completions```)
* Oldest and most widely implemented Open-AI compatible endpoint.
* Represents interaction as a list of role-based messages (```system```, ```user```, ```assistant```).
* Tool calling exists, *but is usually embedded as metadata on an assistant message*. 

#### **Responses** (```/v1/responses```)
* Newer endpoint intended to be the forward path for agentic and multi-modal workflows.
* Represents output as typed "items" (e.g., text blocks, tool calls), which tends to map more cleanly onto agent runners/frontends. 
* Better aligned with where OpenAI is investing long-term, and where format frameworks are standardizing.

### 2. **Why this tutorial will use Responses**
* **More future-proof protocol**: AI work needs to be approached with agent-centric view, and tool calls/other outputs as first-class "items" > having to constantly extrapolate from message metadata. 
* **Cleaner agent plumbing**: it's easier to log, stream, and reason about structured output when the model emits explicit tool-call objects.
* **Ollama supports the key features needed for Agents today**: Support for anything non-stateful in responses API was introduced in version 0.13.3 of Ollama. 

### 3. **What Ollama Responses Support (and don't)**
Ollama's ```/v1/responses``` supports the **non-stateful** flavor of the OpenAI Responses API. Practically, that means: 
#### **Available Today**
* Streaming
* Tools/Function Calling
* Reasoning summaries
* Common model controls like ```temperature```, ```top_p```, ```max_output_tokens```, etc. 

### **Not Available**
* Stateful continuation (```previous_response_id```)
* Conversation handles (```conversation```)
* Truncation strategies (```truncation```)

Basically, that means we have to write some code on our end to handle that stuff, which we will review next.

### 4. **What We Have to Build Ourselves**
Now that we've covered the fact that Ollama is not tracking a conversation on the server, **our client/runner is responsible for this** (tracking the *state*). In time, we will have to implement the following: 

#### A) **Chat Memory/Conversation State**
* Store running chat history (or a summarized version of it).
* Provide that history back to the model each turn (as part of ```input```).
* Optionally add structured memory (facts, preferences, retrieved context) to the prompt.

#### B) **Truncation/Context Window Management**
* When the conversation grows too large, decide what to keep:
    * Drop oldest turns
    * Replace old turns with summaries
    * Keep system instructions + recent turns + retrieved context
* It's ultimately up to us to ensure the requests fit the model's context window, or else you'll run into all sorts of funky output.

#### C) **Tool-Call Reliability and Validation**
* Validate tool-call arguments (types, required fields).
* Handle tool failures and retry logic
* Log tool calls and results for debugging. 

The Agent SDK handles tool *routing* and *invocation*, but not tool *correctness* or *reliability*. You should treat tool functions as production APIs: validate inputs, handle failures explicitly, and log all calls. This remains true even as Ollama’s Responses API gains more features.

#### D) **Persistence**
* If you want memory across sessions, store state in:
    * SQL Database
    * Redis Cache
    * A simple JSONL log (fine for testing)

### 5. **What to Check Later as Ollama (and similar backends) Evolve**
Ollama's OpenAI-compatible APIs are evolving quickly. As support expands, you may be able to remove or simplify parts of your client-side agent infrastructure. In particular, watch for changes in these areas: 

* **Stateful Responses**
    * Support for ```previous_response_id``` and ```conversation```, which would allow the server to manage conversational continuity. 
* **Server-Side Truncation**
    * Support for the ```truncation``` field and documented truncation policies, to reduce/eliminate the need for custom context-window management. 

#### **Note on vLLM** (Ollama alternative)
vLLM also exposes OpenAI-compatible APIs for local LLMs. Like Ollama, these typically operate in a *stateless* mode. As such, the same guidance applies, though it's worth tracking both for updates, developments, and performance considerations. 

## No GPU? Use Ollama Cloud

Everything in this tutorial — including the challenge — is built around Ollama's API. The assumption so far has been that you're running Ollama locally. If you have a capable GPU, that's the fastest path. But **if you don't, Ollama Cloud gives you the exact same API over HTTPS, backed by datacenter hardware, with a generous free tier**.

This is not a workaround. It is the official hosted version of Ollama, maintained by the same team, running the same server code.

### What Ollama Cloud Is
Ollama Cloud (launched late 2025) lets you `ollama run`, `ollama pull`, and make API calls against large open-source models running on NVIDIA-backed cloud infrastructure. From a code perspective, nothing changes except two variables:

| | Local Ollama | Ollama Cloud |
|---|---|---|
| **Base URL** | `http://localhost:11434/v1` | `https://ollama.com/v1` |
| **API key** | any placeholder string | your key from `ollama.com/settings/keys` |
| **`/v1/responses`** | supported (v0.13.3+) | supported (same codebase) |
| **Stateful responses** | not supported | not supported |
| **Rate limits** | none (hardware-bound) | plan-dependent |

The non-stateful constraint we discussed earlier — and all the client-side state management we need to build — applies equally to Ollama Cloud. That's not a cloud limitation; it's a protocol limitation of the current `/v1/responses` spec. The client-side architecture is identical either way.

### Free Tier
Ollama's free tier is $0 and is sufficient to complete this challenge:

| Plan | Cost | Cloud Inference |
|---|---|---|
| **Free** | $0 | Light usage, 1 concurrent cloud model |
| Pro | $20/month | 50x more, 3 concurrent |
| Max | $100/month | 5x Pro, 10 concurrent |

Pricing is a flat subscription — **not per-token** — and all plans include unlimited local inference. The free tier resets on a rolling basis and is designed for experimentation, which is exactly what this challenge requires.

### Setup: Get an API Key
1. Create an account at [ollama.com](https://ollama.com)
2. Go to **Settings → API Keys** (`ollama.com/settings/keys`)
3. Generate a key — it does not expire unless you revoke it
4. Store it in your environment: `export OLLAMA_API_KEY=your_key_here`

### Recommended Cloud Models for Agentic Use
Cloud models are tagged with a `-cloud` suffix in Ollama's registry (when using the CLI), but when calling the API directly you just use the base model name. Good choices for agent/tool-calling workloads:

* **`gpt-oss:20b`** — OpenAI's open-source 20B model; excellent instruction following and tool use, good balance of speed and capability on the free tier
* **`nemotron-3-super`** — NVIDIA's 120B MoE (activates ~12B params); strong agentic reasoning, efficient
* **`devstral-small-2`** — Mistral's 24B model, purpose-built for code and tool-heavy agent workflows
* **`gpt-oss:120b`** — the 120B version if you want maximum capability (Pro tier recommended)

Browse all available cloud models at: `ollama.com/search?c=cloud`

> **Model compatibility note**: Old qwen3 models that emit raw `<think>` tags in their output will NOT work with the Responses API parser. Use qwen3 2507+ versions or any of the models listed above.

### The Code Change Is Literally Two Lines
```python
# Local Ollama:
client = AsyncOpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

# Ollama Cloud:
client = AsyncOpenAI(api_key=os.environ["OLLAMA_API_KEY"], base_url="https://ollama.com/v1")
```

Everything else — `OpenAIResponsesModel`, `Agent`, `Runner`, tool definitions, Sessions — stays the same. The challenge's hard requirement to use `/v1/responses` is fully satisfied by both paths.

## Your First Ollama Agent

### 1. Client, Model, and Agent Setup
Before we run anything, we need to assemble the three foundational pieces the Agent SDK is built around:

1. **Client**: how requests are sent to Ollama (or any model backend)
    * We use ```AsyncOpenAI```, the official OpenAI Python client, to specify that we want to use Ollama with traditional OpenAI API commands (required by SDK).
    * By default, the AgentSDK would otherwise try to use ChatGPT. This lets us override ```base_url```.
    * Model calls are network-bound and often streamed. As such, we use *Async* here to:
        * Stream tokens without blocking
        * Run tools or other tasks *concurrently*
        * Scale cleanly into real applications later

    * We will also set the default client in our tutorial as a best-practice measure. The Agent SDK can create internal model calls behind the scenes (for tools, handoffs, etc.), so setting default client will ensure *every* part of the SDK uses our Ollama-backed client unless explicitly told otherwise.

2. **Model**: an adapter for how the SDK talks to Ollama (or any model backend)
The agents SDK does **not** talk to raw OpenAI endpoints directly. Instead, it uses *model adapters* that describe:
* Which API surface to use (Responses vs. Chat Completions) -- we've already discussed which we prefer.
* How to format inputs
* How to parse outputs and tool calls

    All we have to define for the model is the API Surface, the model we want to use, and the client.

**Note on model choice**: The code cell below defaults to local Ollama. If you're using Ollama Cloud (see the section above), just swap in the two cloud variables — everything else is identical. For the challenge, stronger tool-calling models like `gpt-oss:20b` or `nemotron-3-super` (available on Ollama Cloud's free tier) will give you more reliable agentic behavior than smaller local models.

3. **Agent**: the thing that actually reasons, plans, and responds
An agent is the core abstraction in the SDK. It consists of:
* A **name** -- used for tracing and debugging. Especially useful when you have a multi-agent system!
* **Instructions** -- how should the agent behave, how you want it to respond, etc. 
* Your **model** adapter defined above (which notably includes the actual LLM you want powering this)
* **Tools** it can use -- technically optional, and our first script won't call any. 


In [ ]:
import os
from openai import AsyncOpenAI # This is the official OpenAI Python client. It supports making your model URL configurable, instead of going straight to ChatGPT.
from agents import Agent, Runner # Agent is core building block (Name + LLM + Instructions + Tools). Runner is our orchestration loop.
from agents import set_default_openai_client # Helper function to make the client we define the default for the SDK

from agents import OpenAIResponsesModel # The Responses API you've learned so much about.

# ── Option A: Local Ollama ────────────────────────────────────────────────────
# Requires Ollama v0.13.3+ running locally with a model already pulled.
# Good local models for tool use: glm-4.7-flash, nemotron-3-nano, qwen3:8b-2507+
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
OLLAMA_API_KEY  = 'ollama'           # placeholder; Ollama ignores this for local requests
OLLAMA_MODEL    = 'glm-4.7-flash'

# ── Option B: Ollama Cloud ────────────────────────────────────────────────────
# No GPU required. Free tier is sufficient for this tutorial and the challenge.
# Steps: 1) Sign up at ollama.com  2) Get a key at ollama.com/settings/keys
#        3) export OLLAMA_API_KEY=your_key_here  4) uncomment the three lines below
#
# OLLAMA_BASE_URL = 'https://ollama.com/v1'
# OLLAMA_API_KEY  = os.environ['OLLAMA_API_KEY']   # real key required for cloud
# OLLAMA_MODEL    = 'gpt-oss:20b'   # strong free-tier cloud model for agent/tool use
#                                   # alternatives: nemotron-3-super, devstral-small-2
# ─────────────────────────────────────────────────────────────────────────────

# 1. Create OpenAI-compatible client, which we point to Ollama instead of ChatGPT using AsyncOpenAI. 
client = AsyncOpenAI(
    api_key=OLLAMA_API_KEY,
    base_url=OLLAMA_BASE_URL,  # key detail: route requests to Ollama (local or cloud)
)

# 1.5. Set the default client for the SDK, so anything that needs a client in the future uses this one.
set_default_openai_client(client)

# 2. Define the model we can feed to our Agent block. It requires the client above and the model name.
model = OpenAIResponsesModel(
    model=OLLAMA_MODEL,
    openai_client=client,
)

# 3. Define your Agent (name + instructions + model). We won't incorporate tools yet.
agent = Agent(
    name="ERISA Assistant",
    instructions="You are a sarcastic assistant.",
    model=model,
)

### 2. **Running Your Agent with ```Runner```**
Just to hammer it home, at this point you have already set up:
* ```client``` (AsyncOpenAI pointed at Ollama)
* ```model``` (OpenAIResponsesModel)
* ```agent``` (name + instructions + model [+ tools later])

Now we need the thing that actually **executes** your agent -- enter the ```Runner```.

#### **What ```Runner``` Is**
Your ```Runner``` is your agent orchestration loop -- the engine that *runs* your agent. In plain terms, it
* Sends your input to the agent's model
* Detects tool calls, runs tools, and feeds results back in (once we define some later)
* Returns a structured result object that includes the final text output

#### **Simple Code to Run Agent**

If you were working inside a regular ```.py``` file, the the simplest way to run your agent is:
```python 
prompt = "Why is the sky blue?"
Runner.run_sync(agent, prompt)
print(result.final_output)
```

However, this notebook is running inside Jupyter, which already has an event loop. The Agent SDK intentionally prevents run_sync() from being used when an event loop is active, so trying to run the code above would raise a runtime error.

For that reason, our simple example will use the async runner. Conceptually, it's the same execution path, but it will actually run inside this notebook.


```python
prompt = "Why is the sky blue?"
result = await Runner.run(agent, prompt)
print(result.final_output)
```

#### **Non-Fatal Tracing Messages**
When running the code below, you may see output similar to:
```json
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys.",
    ...etc.
  }
}
```

This message is not related to your model call. The SDK features an optional tracing exporter, which expects an OpenAI API key. Since we are building fully local LLMs here, we do not have or want to insert an OpenAI key. This means we will have to set up our own local tracing solution for full model observability later, using a tool like Langfuse. For now, we can disable these messages with:

```python
from agents import set_tracing_disabled
set_tracing_disabled(True)
```



### **Run Your Agent!**
At this stage, you won't see an output until the entire response has been generated.

In [ ]:
import os
from agents import Runner # We have technically already imported this above, but we'll do it again to be explicit.
from agents import set_tracing_disabled

set_tracing_disabled(True)

prompt = "Write a haiku about dev interns."
result = await Runner.run(agent, prompt) # Agent defined from previous section

print(result.final_output)

Congratulations! You've officially run your first agent! The rest is up to you! :)